<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [179]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [180]:
# TODO (hint: .str.lower().str.contains('pop'))
poptarts = cat[cat['product_name'].str.lower().str.contains('pop', na = 'False')]
print('Pop-Tarts count: ', len(poptarts))
poptarts

Pop-Tarts count:  3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [181]:
# TODO
strawberry = poptarts[poptarts['product_name'].str.lower().str.contains('strawberry', na = False)]
print('Strawberry Pop-Tarts SKUs and names:')
print(strawberry[['sku', 'product_name']])

not_strawberry = poptarts[~poptarts['product_name'].str.lower().str.contains('strawberry', na = False)]
print('Non-Strawberry Pop-Tarts SKUs and names (KEEP SEPARATE): There are NO non-strawberry pop-tarts')
print(not_strawberry[['sku', 'product_name']])

Strawberry Pop-Tarts SKUs and names:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Non-Strawberry Pop-Tarts SKUs and names (KEEP SEPARATE): There are NO non-strawberry pop-tarts
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** There are no other flavors of Pop-Tarts as evidenced by the empty DataFrame

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [182]:
# TODO
initial_row_count = len(cat)

Canonical_mapping = {
    'PT-12': 'SKU#459812',
    'SKU#459812': 'SKU#459812',
    'POPTART-STRAW': 'SKU#459812'
}

cat['canonical'] = cat['sku'].map(Canonical_mapping).fillna(cat['sku'])
cat

,sku,product_name,category,canonical
0,PT-12,pop-tarts strawberry,Breakfast Foods,SKU#459812
1,SKU#459812,Strawberry Pop-Tart,Breakfast,SKU#459812
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,SKU#459812


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [183]:
# TODO
print('Before normalization:')
print(cat['category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()
print('\nAfter normalization:')
print(cat['category'].value_counts(dropna=False))

Before normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [184]:
# TODO
cat['normalized'] = (cat['product_name'].str.lower()
                     .str.replace('[^a-z]', '', regex = True)
                     .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
poptartsstrawberry,"[PT-12, POPTART-STRAW]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [185]:
# TODO
assert cat['canonical'].notna().all()
assert len(cat) == initial_row_count
print('Mapping is safe: produced no nulls and did not change the number of rows')

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

Two product families beyond Pop-Tarts that appear fragmented include bottled water and flashlights (WATER-24 and FLASH-24 were seen present in the lecture content) through different SKUs and spelling variations. I would group these together by mapping them to one canonical SKU, such as mapping all water and bottled water 24 packs into BTL-WATER-24, and keeping other product types of sizes as separate SKUs.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Consolidations transitions from merely data-cleaning into business highly important business decisions when differences change from spelling errors of the same proper identity into real-life differences between product types of sizes. For example, consolidation could be necessary between 6 and 24 packs of bottles or Strawberry Pop-Tarts. Whether or not to combine them under one canonical SKU would be a difficult, uninformed decision, so I would ask the store's inventory manager or my higher up positions about what the goals are with the SKU presence (whether combining would be better to track over all demand or separating would be better for specific inventories.